# Notebook 02 - Features, Embeddings, and Similarity

This notebook starts from the cleaned file produced by Notebook 01:

`contracts_ie_clean.csv`

It creates reusable text features for the modeling notebook:

- TF-IDF matrix and fitted vectorizer
- Multilingual sentence embeddings
- Cross-buyer cosine similarity scores for copy-paste detection
- A feature CSV with `copy_paste_description` and optional NER entity-linking features

Designed to run in Google Colab temporary storage or local Jupyter. In Colab, upload `contracts_ie_clean.csv` when prompted.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Running in:', 'Google Colab' if IN_COLAB else 'Local Jupyter')

if IN_COLAB:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'sentence-transformers', 'scikit-learn', 'scipy', 'joblib', 'tqdm'
    ], check=True)

DATA_DIR = Path.cwd()
OUTPUT_DIR = DATA_DIR / 'outputs_02'
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH = DATA_DIR / 'contracts_ie_clean.csv'
print('DATA_DIR  :', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# Colab convenience: upload the cleaned CSV directly into temporary storage.
if IN_COLAB and not CSV_PATH.exists():
    from google.colab import files
    print('Upload contracts_ie_clean.csv')
    uploaded = files.upload()
    if 'contracts_ie_clean.csv' not in uploaded:
        raise FileNotFoundError('Please upload a file named contracts_ie_clean.csv')

if not CSV_PATH.exists():
    raise FileNotFoundError(f'Could not find {CSV_PATH}. Put contracts_ie_clean.csv next to this notebook.')

## 1. Load and Validate Data

In [ ]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f'Loaded {len(df):,} rows and {df.shape[1]} columns')

required_cols = [
    'contract_id', 'title', 'description', 'buyer_id', 'buyer_name',
    'single_bid', 'short_tender_period', 'winner_concentration'
]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns from Notebook 01 output: {missing}')

df[required_cols].head(3)

## 2. Build Text Field

In [ ]:
df['text_for_model'] = df['description'].fillna('').astype(str).str.strip()
empty_description = df['text_for_model'].eq('')
df.loc[empty_description, 'text_for_model'] = df.loc[empty_description, 'title'].fillna('').astype(str).str.strip()

texts = df['text_for_model'].tolist()

print(f'Text rows              : {len(texts):,}')
print(f'Empty after title fill : {sum(t == "" for t in texts):,}')
print('\nSample text:')
print(texts[0][:500])

## 3. TF-IDF Baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import joblib
import time

tfidf = TfidfVectorizer(
    sublinear_tf=True,
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=3,
    strip_accents='unicode',
    analyzer='word'
)

t0 = time.time()
X_tfidf = tfidf.fit_transform(texts)
elapsed = time.time() - t0

print(f'TF-IDF shape       : {X_tfidf.shape}')
print(f'Non-zero values    : {X_tfidf.nnz:,}')
print(f'Sparsity           : {100 * (1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])):.2f}%')
print(f'Completed in       : {elapsed:.1f} seconds')

save_npz(OUTPUT_DIR / 'tfidf_matrix.npz', X_tfidf)
joblib.dump(tfidf, OUTPUT_DIR / 'tfidf_vectorizer.joblib')
print('Saved TF-IDF artifacts')

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
mean_scores = np.asarray(X_tfidf.mean(axis=0)).ravel()
top_idx = mean_scores.argsort()[-20:][::-1]

pd.DataFrame({
    'term': feature_names[top_idx],
    'mean_tfidf': mean_scores[top_idx]
})

## 4. Sentence Embeddings

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Set SAMPLE_SIZE to a smaller number for a quick CPU test, or None for the full dataset.
SAMPLE_SIZE = None

if SAMPLE_SIZE is None:
    encode_df = df.copy()
else:
    encode_df = df.head(SAMPLE_SIZE).copy()

encode_texts = encode_df['text_for_model'].tolist()
print(f'Device         : {DEVICE}')
print(f'Rows to encode : {len(encode_texts):,}')

encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)
print('Embedding dim  :', encoder.get_sentence_embedding_dimension())

In [ ]:
batch_size = 128 if DEVICE == 'cuda' else 32

t0 = time.time()
embeddings = encoder.encode(
    encode_texts,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')
elapsed = time.time() - t0

print(f'Embeddings shape : {embeddings.shape}')
print(f'Dtype            : {embeddings.dtype}')
print(f'Completed in     : {elapsed / 60:.1f} minutes')

np.save(OUTPUT_DIR / 'embeddings.npy', embeddings)
encode_df[['contract_id']].to_csv(OUTPUT_DIR / 'contract_ids.csv', index=False)
print('Saved embeddings.npy and contract_ids.csv')

In [ ]:
norms = np.linalg.norm(embeddings[:min(1000, len(embeddings))], axis=1)
print(f'Norm check: min={norms.min():.6f}, max={norms.max():.6f}, mean={norms.mean():.6f}')

## 5. Cross-Buyer Similarity Scan

In [ ]:
from tqdm.auto import tqdm

SIMILARITY_THRESHOLD = 0.92
BLOCK_SIZE = 2000

n = len(embeddings)
buyer_ids = encode_df['buyer_id'].fillna('__missing_buyer__').astype(str).values
contract_ids = encode_df['contract_id'].astype(str).values

copy_paste_flags = np.zeros(n, dtype=np.int8)
flagged_pairs = []

print(f'Scanning {n:,} contracts in {BLOCK_SIZE}-row blocks')
print(f'Threshold: cosine similarity >= {SIMILARITY_THRESHOLD}')

t0 = time.time()
for i in tqdm(range(0, n, BLOCK_SIZE)):
    block_a = embeddings[i:i + BLOCK_SIZE]
    buyers_a = buyer_ids[i:i + BLOCK_SIZE]

    for j in range(i, n, BLOCK_SIZE):
        block_b = embeddings[j:j + BLOCK_SIZE]
        buyers_b = buyer_ids[j:j + BLOCK_SIZE]

        sim = block_a @ block_b.T
        if i == j:
            np.fill_diagonal(sim, 0.0)

        different_buyer = buyers_a[:, None] != buyers_b[None, :]
        rows, cols = np.where((sim >= SIMILARITY_THRESHOLD) & different_buyer)

        for r, c in zip(rows, cols):
            a = i + r
            b = j + c
            if a >= b:
                continue
            copy_paste_flags[a] = 1
            copy_paste_flags[b] = 1
            flagged_pairs.append({
                'contract_id_a': contract_ids[a],
                'contract_id_b': contract_ids[b],
                'buyer_id_a': buyer_ids[a],
                'buyer_id_b': buyer_ids[b],
                'cosine_sim': round(float(sim[r, c]), 5)
            })

elapsed = time.time() - t0
print(f'Finished in {elapsed / 60:.1f} minutes')
print(f'Flagged pairs            : {len(flagged_pairs):,}')
print(f'Unique contracts flagged : {copy_paste_flags.sum():,}')

In [ ]:
similarity_df = pd.DataFrame(flagged_pairs)

if not similarity_df.empty:
    id_to_buyer_name = dict(zip(encode_df['contract_id'].astype(str), encode_df['buyer_name'].fillna('').astype(str)))
    id_to_title = dict(zip(encode_df['contract_id'].astype(str), encode_df['title'].fillna('').astype(str)))

    similarity_df = similarity_df.sort_values('cosine_sim', ascending=False).reset_index(drop=True)
    similarity_df['buyer_name_a'] = similarity_df['contract_id_a'].map(id_to_buyer_name)
    similarity_df['buyer_name_b'] = similarity_df['contract_id_b'].map(id_to_buyer_name)
    similarity_df['title_a'] = similarity_df['contract_id_a'].map(id_to_title)
    similarity_df['title_b'] = similarity_df['contract_id_b'].map(id_to_title)

similarity_df.to_csv(OUTPUT_DIR / 'similarity_scores.csv', index=False)
print(f'Saved {len(similarity_df):,} rows to similarity_scores.csv')
similarity_df.head(10)

## 6. Save Feature File for Notebook 03

In [ ]:
features_df = encode_df.copy()
features_df['copy_paste_description'] = copy_paste_flags

# Optional NER integration: merge engineered entity-linking features when available.
# Expected file from the NER workflow: ner_features (full).csv
ner_path = DATA_DIR / 'ner_features (full).csv'
if not ner_path.exists():
    ner_path = DATA_DIR / 'ner_features.csv'

if ner_path.exists():
    import ast

    def list_count(value):
        if pd.isna(value):
            return 0
        try:
            parsed = ast.literal_eval(str(value))
            return len(parsed) if isinstance(parsed, list) else 0
        except Exception:
            return 0

    ner_df = pd.read_csv(ner_path)
    ner_df['shared_address_flag'] = pd.to_numeric(
        ner_df['shared_address_flag'], errors='coerce'
    ).fillna(0).astype(int)
    ner_df['num_extracted_companies'] = ner_df['extracted_companies'].apply(list_count)
    ner_df['num_extracted_locations'] = ner_df['extracted_locations'].apply(list_count)
    ner_df['has_extracted_company'] = (ner_df['num_extracted_companies'] > 0).astype(int)
    ner_df['has_extracted_location'] = (ner_df['num_extracted_locations'] > 0).astype(int)

    features_df = features_df.merge(ner_df, on='contract_id', how='left', validate='one_to_one')
    for col in ['shared_address_flag', 'num_extracted_companies', 'num_extracted_locations',
                'has_extracted_company', 'has_extracted_location']:
        features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0).astype(int)
    for col in ['extracted_companies', 'extracted_locations']:
        features_df[col] = features_df[col].fillna('[]')
    print(f'Merged NER features from: {ner_path.name}')
    print(f'shared_address_flag rate: {features_df["shared_address_flag"].mean() * 100:.2f}%')
else:
    print('NER feature file not found; saving features without NER columns.')

feature_path = OUTPUT_DIR / 'contracts_ie_features.csv'
features_df.to_csv(feature_path, index=False)

print(f'Saved feature file: {feature_path}')
print(f'Rows: {len(features_df):,}')
print(f'copy_paste_description rate: {features_df["copy_paste_description"].mean() * 100:.2f}%')


In [ ]:
deliverables = [
    'tfidf_matrix.npz',
    'tfidf_vectorizer.joblib',
    'embeddings.npy',
    'contract_ids.csv',
    'similarity_scores.csv',
    'contracts_ie_features.csv'
]

print('Notebook 02 deliverables')
print('-' * 60)
for name in deliverables:
    path = OUTPUT_DIR / name
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    status = 'OK' if path.exists() else 'MISSING'
    print(f'{name:<30} {status:<8} {size_mb:>8.2f} MB')